# Homework 3: Percentiles, spread, and records

Covers **Lecture 6** (percentiles, quartiles, the interquartile range, histograms) and
**Lecture 7** (lists, tuples, dictionaries).

## Submission instructions

Upload the `ipynb` file to Canvas:
> File -> Download -> ipynb -> upload to Canvas (like any other file)

The `ipynb` file type is the only accepted format.

Save a copy before beginning. Reloading or disconnecting can revert the notebook to the
instructor copy.

# Problem 0 (0 pts): generative AI usage statement

As you work on this assignment, feel free to use generative AI tools to help you learn,
understand, and debug Python code. In particular, you could get hints or conceptual guidance in
the implementation you write yourself.

However, you must clearly disclose and cite all use of AI. You must include:
1. The name(s) of the AI tool(s) used.
2. The specific prompt(s) you used to generate the content.
3. A description of how you used the output and what edits or additions you made to integrate it
   into your own work.

You are fully responsible for the final submitted work -- critically evaluate, fact-check, and
verify all AI-generated content for validity. Failure to properly cite and disclose AI use
constitutes plagiarism under Penn State's Academic Integrity policy.

Write your disclosure (or "I did not use an AI tool for this assignment") in the cell below.

*your disclosure here*

Lecture 7 held one study from the tetrahedrite database in a dictionary. In this
assignment, use the whole collection to assess typical zT, the variation among samples,
and the information lost when results from 63 studies are reduced to one summary number.

In [1]:
#@title Load the tetrahedrite thermoelectric database (click ▶ to run, data loading, not a learning objective) { display-mode: "form" }
import os

import numpy as np
import pandas as pd

_file = 'thermoelectrics.csv'
_github = f'https://raw.githubusercontent.com/wfreinhart/matse219/main/datasets/{_file}'
_local = next(
    (p for p in (f'datasets/{_file}', f'../datasets/{_file}', f'../../datasets/{_file}',
                 f'../../../datasets/{_file}')
     if os.path.exists(p)),
    None,
)

try:
    _table = pd.read_csv(_local if _local else _github)
except Exception as e:
    raise RuntimeError(
        f"Could not load '{_file}'. If you are in Colab, check your internet "
        f"connection and that the file exists at {_github}"
    ) from e

_rows = _table.dropna(subset=['zT_max'])

zt = _rows['zT_max'].to_numpy()

print(f'Loaded {len(_table)} tetrahedrite samples from {_file}')
print(f'  {len(_table) - len(_rows)} samples have no reported zT and are left out')
print(f'  zt: {len(zt)} values (the highest zT each sample reached)')

Loaded 279 tetrahedrite samples from thermoelectrics.csv
  1 samples have no reported zT and are left out
  zt: 278 values (the highest zT each sample reached)


`zt` holds the highest figure of merit reached by each of 278 samples. zT is
dimensionless and bigger is better; a sample near 1 competes with the commercial
tellurides, and a sample near zero converts almost nothing.

# Problem 1 (25 pts): `np.percentile` and order statistics

(a) Print the smallest and largest values in `zt`, and the number of samples.

(b) The nine numbers below are real zT values, one taken from each ninth of the sorted
database.

```python
nine = [0.69, 0.0, 0.54, 0.87, 0.37, 0.75, 0.59, 0.8, 0.64]
```

Sort them **by hand** and write the sorted list in a text cell. Then, using only
positions in your sorted list, write down the median and the first and third quartiles.

Now run `np.percentile(nine, [25, 50, 75])` and compare with your three answers.

> For nine values the quartile positions land exactly on data points, so NumPy returns
> the same numbers you found by hand. For most other lengths it interpolates between
> neighbours, and hand methods disagree with each other too. You are not responsible for
> the rule. You are responsible for knowing that "the quartile" is a convention.

(c) Print the 10th, 50th and 90th percentiles of the full `zt` array in one call.

Then, in 2-3 sentences: the 10th percentile is far below the median while the 90th is
close to it. Describe where the samples are concentrated and explain the pattern among
the values at the low end.

> The near-zero samples are not mistakes. They are compositions whose doping destroyed
> the figure of merit, which is a result worth publishing.

# Problem 2 (20 pts): `np.std`, the interquartile range, and the histogram recipe

(a) Print the mean, the median, the standard deviation, and the interquartile range
(the 75th percentile minus the 25th) of `zt`. Label each and show two decimals.

> Mean and median are both "middle" numbers, and standard deviation and IQR are both
> "spread" numbers. Notice whether each pair agrees before you go on.

(b) Plot a histogram of `zt` using the recipe below, changing only the data and the
x-axis label.

```python
import matplotlib.pyplot as plt

plt.hist(zt, bins=30)
plt.xlabel('...')
plt.ylabel('number of samples')
plt.show()
```

> This is a recipe to copy, not syntax to memorize. Matplotlib gets taken apart properly
> in a later lecture.

(c) In L06 the efficiency loss had a mean well above its median. Here the mean sits
slightly below it instead.

In 3-4 sentences, and using the histogram you just drew: which side of this distribution
has the long tail, which of the two summaries moved because of it, and would you report
the standard deviation or the IQR to a colleague who asked how much tetrahedrites differ
from one another? Either choice can earn full marks; the reasoning is graded.

# Problem 3 (25 pts): `KeyError`, `IndexError`, and dictionaries as records

In Lecture 7, you built records for whole studies. These problems work one level down,
on single samples and on the dopant families they belong to.

(a) **Before running it**, read the cell below and write in a text cell which error it
raises and which line causes it. Then run it, say whether you were right, and fix it by
printing something the record does contain.

```python
family = {'name': 'Te', 'n_samples': 16, 'best_zt': 0.92}
print(family['median_zt'])
```

(b) Same procedure for this cell: predict the error first, then run it, then fix it so
it prints the last entry **without changing the list**.

```python
families = ['Se', 'Co', 'Fe']
print(families[3])
```

(c) One sample in the database comes from Kwak 2021: composition `Cu12Sb3.8Te0.2S13`,
Te-doped, and it reached zT 0.70 at 450 C.

Build a dictionary called `sample` holding those five facts, choosing a sensible key for
each. Then print `sample.keys()` and `len(sample)`.

(d) Two of the five values you stored describe *this specimen*, and the rest would be the
same for every Te-doped sample in the study.

In 3-4 sentences: which are which, and what could go wrong later if someone copied this
record for the study's next sample and changed only the zT?

# Problem 4 (30 pts): lists of dictionaries and `.append()`

Samples in this database are grouped by what the tetrahedrite was doped with. The four
dictionaries below summarise four dopant families: how many samples used that dopant, the
best zT any of them reached, and the median zT of the family.

```python
families = [
    {'dopant': 'Se', 'n_samples': 14, 'best_zt': 1.10, 'median_zt': 0.78},
    {'dopant': 'Mn', 'n_samples': 16, 'best_zt': 1.13, 'median_zt': 0.64},
    {'dopant': 'Co', 'n_samples': 17, 'best_zt': 0.98, 'median_zt': 0.72},
    {'dopant': 'Fe', 'n_samples': 14, 'best_zt': 0.83, 'median_zt': 0.51},
]
```

(a) Print the dopant and the best zT of the **first** and the **last** family in the
list, indexing into the list and then into the dictionary.

> You would write that line 27 times to cover every dopant family in the database. That
> is the problem loops solve, and they are two lectures away. Do not try to solve it now.

(b) Build a fifth record for the Ni family, which has 18 samples, a best zT of 0.95 and a
median of 0.67. `.append()` it to `families`, then print `len(families)` and the dopant
in the last record.

(c) Print the best zT and the median zT of the Mn family, then the same two numbers for
the Se family, by indexing into `families`.

(d) Mn holds the highest single zT in the entire database, and Se does not. Se has the
higher median of the two.

Answer both parts in 4-6 sentences total.

1. You have to pick one of those two dopants to pursue in your own lab, and you can make
   a handful of samples. State which dopant you would pursue. Then say which summary --
   its best zT or its median -- better describes what you should expect from a sample you
   make yourself, and why. Either dopant choice can earn full marks; the reasoning is
   what is graded.
2. These five families hold 79 of the 278 samples between them. What would you need in
   order to work out the median for a family yourself, rather than being given it, and
   why can you not do it with what you know so far?

> A median is a middle, not a promise. A family of 14 samples and a family of 18 are not
> equally well described by one number each.